# Mature-series baseline benchmark

Evaluate the three baselines used to calibrate maturity in `02_01`. A product-store series enters an origin only when its history strictly before that origin satisfies the shared maturity rule.

The primary benchmark is the same-weekday moving average over the last four matching weekdays, with the recent 28-observation daily mean as fallback.

In [1]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

PROJECT_ROOT = next(
    (path.resolve() for path in [Path('../..'), Path('..'), Path('.')]
     if (path / 'src').exists()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Could not find the project root containing src/.')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.models.benchmark import (
    MODEL_COLUMNS, PRIMARY_MODEL, load_benchmark_design,
    segment_wape, summarize_models,
)
from src.models.results import load_benchmark_result

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 180)


## Design and origin population

In [2]:
design = load_benchmark_design()
design_table = pd.DataFrame([{
    'data directory': design.data_dir,
    'minimum active days': design.min_active_days,
    'maximum origins': design.max_origins,
    'first origin': design.first_origin.date(),
    'origin spacing days': design.origin_spacing_days,
    'forecast horizon days': design.forecast_horizon_days,
}])
display(design_table)

result = load_benchmark_result(design)
display(result.data_audit)
print(f'Number of evaluated origins: {len(result.origin_summary):,}')
display(result.origin_summary.style.format({
    'known_series': '{:,.0f}',
    'assessed_series': '{:,.0f}',
    'assessed_fcm_series': '{:,.0f}',
    'assessed_pseudo_series': '{:,.0f}',
    'evaluated_series': '{:,.0f}',
}))


,data directory,minimum active days,maximum origins,first origin,origin spacing days,forecast horizon days
0,/Users/vlada/UNI/SoSe2026/ba/ba_code/data/proc...,28,20,2026-03-02,7,7


,source_rows,series,first_date,last_date,overlapping_source_rows,null_active_flags,active_rows_with_closure_reason,closed_rows_without_reason,closed_rows_with_demand,changing_source_series,changing_category_series
0,20278470,22009,2023-07-22,2026-07-21,0.0,0.0,0.0,0.0,0.0,0,0


Number of evaluated origins: 20


,origin,known_series,assessed_series,assessed_fcm_series,assessed_pseudo_series,evaluated_series
0,2026-03-02 00:00:00,"21,695","21,443","2,838","18,605","21,443"
1,2026-03-09 00:00:00,"21,702","21,590","2,978","18,612","21,590"
2,2026-03-16 00:00:00,"21,706","21,614","2,993","18,621","21,614"
3,2026-03-23 00:00:00,"21,825","21,628","3,004","18,624","21,628"
4,2026-03-30 00:00:00,"21,839","21,690","3,004","18,686","21,690"
5,2026-04-06 00:00:00,"21,843","21,697","3,006","18,691","21,697"
6,2026-04-13 00:00:00,"21,883","21,702","3,007","18,695","21,702"
7,2026-04-20 00:00:00,"21,910","21,711","3,008","18,703","21,711"
8,2026-04-27 00:00:00,"21,916","21,828","3,011","18,817","21,828"
9,2026-05-04 00:00:00,"21,923","21,839","3,015","18,824","21,839"


Each origin is evaluated independently. Counts can increase over time because a series is admitted from its first mature origin onward; no future maturity status is carried backward.

In [3]:
model_summary = summarize_models(result.forecasts)
model_summary.insert(1, 'model_label', model_summary['model'].map(MODEL_COLUMNS))
model_summary.insert(2, 'primary', model_summary['model'].eq(PRIMARY_MODEL))
display(model_summary.style.format({
    'pooled_wape': '{:.2%}',
    'relative_bias': '{:+.2%}',
    'forecast_to_actual_ratio': '{:.3f}',
    'median_series_wape': '{:.2%}',
    'seasonal_mase': '{:.3f}',
    'mae_kg': '{:.3f}',
    'mean_bias_kg': '{:+.3f}',
    'actual_kg': '{:,.1f}',
    'forecast_kg': '{:,.1f}',
    'n_series': '{:,.0f}',
    'n_origins': '{:,.0f}',
    'n_forecast_rows': '{:,.0f}',
    'seasonal_mase_valid_share': '{:.1%}',
}))

,model,model_label,primary,pooled_wape,relative_bias,forecast_to_actual_ratio,median_series_wape,seasonal_mase,mae_kg,mean_bias_kg,actual_kg,forecast_kg,n_series,n_origins,n_forecast_rows,seasonal_mase_valid_share
0,recent_mean,Recent mean (28 observations),False,97.93%,-17.49%,0.825,132.23%,0.894,0.996,-0.178,"2,552,925.9","2,106,495.5","21,937",20,"2,511,229",100.0%
1,same_weekday_moving_average,Same-weekday moving average (4 occurrences; recent-mean fallback),True,102.59%,+0.52%,1.005,141.91%,0.932,1.043,+0.005,"2,552,925.9","2,566,096.8","21,937",20,"2,511,229",100.0%
2,occurrence_x_positive_quantity,Occurrence x positive quantity,False,112.10%,+3.59%,1.036,143.55%,0.970,1.140,+0.036,"2,552,925.9","2,644,565.2","21,937",20,"2,511,229",100.0%


## Primary

Pooled WAPE and volume calibration are the primary benchmark readout. Relative bias is `(forecast - actual) / actual`; the forecast-to-actual ratio is the same calibration expressed around 1.

In [4]:
primary = model_summary.loc[
    model_summary['model'].eq(PRIMARY_MODEL),
    [
        'model_label',
        'pooled_wape',
        'relative_bias',
        'forecast_to_actual_ratio',
        'actual_kg',
        'forecast_kg',
        'n_series',
        'n_origins',
    ],
]
display(primary.style.format({
    'pooled_wape': '{:.2%}',
    'relative_bias': '{:+.2%}',
    'forecast_to_actual_ratio': '{:.3f}',
    'actual_kg': '{:,.1f}',
    'forecast_kg': '{:,.1f}',
    'n_series': '{:,.0f}',
    'n_origins': '{:,.0f}',
}))

,model_label,pooled_wape,relative_bias,forecast_to_actual_ratio,actual_kg,forecast_kg,n_series,n_origins
1,Same-weekday moving average (4 occurrences; recent-mean fallback),102.59%,+0.52%,1.005,"2,552,925.9","2,566,096.8","21,937",20


## Secondary

The secondary view describes typical-series error, scale-free weekly-seasonal error, error in kilograms, and pooled performance within operational segments. Seasonal MASE uses the pre-origin mean absolute difference at a seven-calendar-day lag.

In [5]:
secondary = model_summary.loc[
    model_summary['model'].eq(PRIMARY_MODEL),
    [
        'model_label',
        'median_series_wape',
        'seasonal_mase',
        'mae_kg',
        'mean_bias_kg',
        'seasonal_mase_valid_share',
    ],
]
display(secondary.style.format({
    'median_series_wape': '{:.2%}',
    'seasonal_mase': '{:.3f}',
    'mae_kg': '{:.3f}',
    'mean_bias_kg': '{:+.3f}',
    'seasonal_mase_valid_share': '{:.1%}',
}))

,model_label,median_series_wape,seasonal_mase,mae_kg,mean_bias_kg,seasonal_mase_valid_share
1,Same-weekday moving average (4 occurrences; recent-mean fallback),141.91%,0.932,1.043,+0.005,100.0%


In [6]:
primary_forecasts = result.forecasts[result.forecasts['model'].eq(PRIMARY_MODEL) & result.forecasts['is_active']].copy()
segment_format = {
    'pooled_wape': '{:.2%}',
    'relative_bias': '{:+.2%}',
    'forecast_to_actual_ratio': '{:.3f}',
    'actual_kg': '{:,.1f}',
    'n_series': '{:,.0f}',
    'n_origins': '{:,.0f}',
}
print('Segment-level WAPE by sourcing group')
display(segment_wape(primary_forecasts, 'sourcing_group').style.format(segment_format))
print('Segment-level WAPE by merchandise category')
display(segment_wape(primary_forecasts, 'category_id').style.format(segment_format))
print('All-model results by sourcing group and merchandise category')
combined_segments = segment_wape(
    result.forecasts[result.forecasts['is_active']], ['sourcing_group', 'category_id']
)
combined_segments.insert(
    3, 'model_label', combined_segments['model'].map(MODEL_COLUMNS)
)
display(combined_segments.style.format(segment_format))

Segment-level WAPE by sourcing group


,sourcing_group,model,pooled_wape,relative_bias,forecast_to_actual_ratio,actual_kg,n_series,n_origins
0,FCM,same_weekday_moving_average,106.87%,+6.52%,1.065,"60,577.5","3,074",20
1,Pseudo,same_weekday_moving_average,102.49%,+0.37%,1.004,"2,492,348.4","18,863",20


Segment-level WAPE by merchandise category


,category_id,model,pooled_wape,relative_bias,forecast_to_actual_ratio,actual_kg,n_series,n_origins
0,890,same_weekday_moving_average,102.62%,+0.43%,1.004,"2,493,267.0","18,937",20
1,900,same_weekday_moving_average,101.39%,+4.06%,1.041,"59,659.0","3,000",20


All-model results by sourcing group and merchandise category


,sourcing_group,category_id,model,model_label,pooled_wape,relative_bias,forecast_to_actual_ratio,actual_kg,n_series,n_origins
0,FCM,890,occurrence_x_positive_quantity,Occurrence x positive quantity,204.84%,+56.91%,1.569,"3,840.1",318,20
1,FCM,890,recent_mean,Recent mean (28 observations),186.13%,+29.86%,1.299,"3,840.1",318,20
2,FCM,890,same_weekday_moving_average,Same-weekday moving average (4 occurrences; recent-mean fallback),198.85%,+60.46%,1.605,"3,840.1",318,20
3,FCM,900,occurrence_x_positive_quantity,Occurrence x positive quantity,100.14%,+3.68%,1.037,"56,737.4","2,756",20
4,FCM,900,recent_mean,Recent mean (28 observations),94.03%,-15.31%,0.847,"56,737.4","2,756",20
5,FCM,900,same_weekday_moving_average,Same-weekday moving average (4 occurrences; recent-mean fallback),100.64%,+2.87%,1.029,"56,737.4","2,756",20
6,Pseudo,890,occurrence_x_positive_quantity,Occurrence x positive quantity,112.23%,+3.48%,1.035,"2,489,426.8","18,619",20
7,Pseudo,890,recent_mean,Recent mean (28 observations),97.88%,-17.63%,0.824,"2,489,426.8","18,619",20
8,Pseudo,890,same_weekday_moving_average,Same-weekday moving average (4 occurrences; recent-mean fallback),102.47%,+0.34%,1.003,"2,489,426.8","18,619",20
9,Pseudo,900,occurrence_x_positive_quantity,Occurrence x positive quantity,112.31%,+25.84%,1.258,"2,921.6",244,20
